In [1]:
!pip install -q "pymilvus[milvus_lite]" opik

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.6/269.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.1/668.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.7

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient

from openai import OpenAI

import opik
from opik.evaluation.metrics import Hallucination, AnswerRelevance

import warnings
warnings.filterwarnings('ignore')

In [ ]:
os.environ['HF_TOKEN'] = ''
os.environ['OPIK_API_KEY'] = ''
os.environ['OPENAI_API_KEY'] = ''

print('Environment Configured')

Environment Configured


# Loading Dataset

In [4]:
# loading dataset from huggingface

dataset = load_dataset('m-ric/huggingface_doc') # list of dictionaries

train_split = dataset['train']
sample_row = train_split[0]
sample_text = sample_row['text'].replace('\n', ' ')

print('----Dataset Loaded----')
print(f"Number of rows in the dataset: {train_split.num_rows} rows")
print(f"Columns present in the dataset: {train_split.features}")

print()
print('Sample row')
print(f"Text: {sample_text[:500]}")
print(f"Source: {sample_row['source'][:500]}")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

huggingface_doc.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

----Dataset Loaded----
Number of rows in the dataset: 2647 rows
Columns present in the dataset: {'text': Value('string'), 'source': Value('string')}

Sample row
Text:  Create an Endpoint  After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification.   ## 1. Enter the Hugging Face Repository ID and your desired endpoint name:  <img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-docu
Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


In [5]:
# converting huggingface object into a python list

documents = []

for item in dataset['train']:
  documents.append({
      'text': item['text'],
      'source': item['source']
  })

print(f'Extracted {len(documents)} documents from the dataset')

Extracted 2647 documents from the dataset


# Chunking

In [6]:
# we will be using chunking with overlap

def chunk_document(text: str, chunk_size: int = 1000, chunk_overlap: int=200) -> List[str]:

  chunks = []

  if not text.strip():
    return []

  if len(text) <= chunk_size:
    return [text.strip()]

  step = chunk_size - chunk_overlap
  if step <= 0:
    raise ValueError('Chunk Overlap must be strictly less than Chunk Size')

  start = 0

  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start : end])

    # stop once a chunk reaches the end of the text. Without this the loop adds one more
    # short chunk that is already fully contained in the overlap of the previous chunk.
    if end >= len(text):
      break

    start += step

  return chunks


def chunk_all_documents(documents: List[Dict], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Dict]:

  all_chunks = []
  chunk_id = 0

  for document in documents:
    text = document['text']
    chunks = chunk_document(text, chunk_size, chunk_overlap)

    for chunk in chunks:
      all_chunks.append({
          'chunk': chunk,
          'source': document['source'],
          'chunk_id': chunk_id
      })

      chunk_id += 1

  return all_chunks

In [7]:
# Test your chunking implementation

test_text = "A" * 2500  # 2500 characters
test_chunks = chunk_document(test_text, chunk_size=1000, chunk_overlap=200)

print(f"Test: 2500 char text with chunk_size=1000, overlap=200")
print(f"Expected chunks: ~4")
print(f"Your chunks: {len(test_chunks)}")

if len(test_chunks) >= 3 and len(test_chunks) <= 5:
    print("✅ Chunking test passed!")
else:
    print("❌ Check your chunking implementation")

Test: 2500 char text with chunk_size=1000, overlap=200
Expected chunks: ~4
Your chunks: 3
✅ Chunking test passed!


In [8]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = chunk_all_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f'Created {len(chunks)} chunks from {len(documents)} documents')
print(f'Average chunks per document: {len(chunks) / len(documents):.2f}')

if chunks:
  print()
  print('-----Sample Chunk-----')
  print(f"Chunk ID: {chunks[0]['chunk_id']}")
  print(f"Chunk Text: {chunks[0]['chunk']}")
  print(f"Chunk Source: {chunks[0]['source']}")

Created 27437 chunks from 2647 documents
Average chunks per document: 10.37

-----Sample Chunk-----
Chunk ID: 0
Chunk Text:  Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />

## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.

<img src="https://raw.githubusercontent.c

# Embeddings

In [9]:
# defining the embeddings model

EMBEDDING_MODEL = 'BAAI/bge-small-en-v1.5'
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Loaded embedding model: {EMBEDDING_MODEL}")

# Test embedding
test_embedding = embedding_model.encode(["This is a test"], normalize_embeddings=True)
EMBEDDING_DIM = len(test_embedding[0])
print(f"Embedding dimension: {EMBEDDING_DIM}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


In [10]:
# writing a function to generate embeddings

def generate_embeddings(texts: List[str], model: SentenceTransformer, batch_size: int = 32) -> List[List[float]]:

  # a single encode call shows one progress bar instead of one per batch,
  # and lets sentence-transformers group texts of similar length, which is faster
  embeddings = model.encode(
      texts,
      batch_size = batch_size,
      normalize_embeddings = True,
      show_progress_bar = len(texts) > batch_size
  )

  return embeddings.tolist()

In [11]:
# test to check if the embeddings generated are correct

test_texts = ["Hello world", "This is a test", "RAG is cool"]
test_embeddings = generate_embeddings(test_texts, embedding_model)

print(f"Generated {len(test_embeddings)} embeddings")
print(f"Embedding dimension: {len(test_embeddings[0]) if test_embeddings else 0}")

if len(test_embeddings) == 3 and len(test_embeddings[0]) == 384:
    print("✅ Embedding generation test passed!")
else:
    print("❌ Check your embedding implementation")

Generated 3 embeddings
Embedding dimension: 384
✅ Embedding generation test passed!


In [12]:
chunk_texts = [chunk['chunk'] for chunk in chunks]
embeddings = generate_embeddings(chunk_texts, embedding_model)

Batches:   0%|          | 0/858 [00:00<?, ?it/s]

In [13]:
if embeddings:
  print(f'Embeddings Dimension: {len(embeddings[0])}')
  print(f'First 10 values of Sample Embedding: {embeddings[0][:10]}')

Embeddings Dimension: 384
First 10 values of Sample Embedding: [-0.07532959431409836, -0.027507992461323738, -0.03995613381266594, -0.040492136031389236, 0.033340033143758774, 0.04296518489718437, -0.043336279690265656, -0.04493821784853935, -0.05554318055510521, 0.02672027423977852]


# Vector Store

In [14]:
MILVUS_DB_PATH = './hf_docs_milvus.db'
milvus_client = MilvusClient(uri = MILVUS_DB_PATH)

COLLECTION_NAME = "hf_documentation"

print(f"Milvus client initialized with database: {MILVUS_DB_PATH}")

Milvus client initialized with database: ./hf_docs_milvus.db


In [15]:
# setting up the vector store

def setup_milvus_collection(client : MilvusClient, collection_name: str, embedding_dim: int):

  if client.has_collection(collection_name):
    client.drop_collection(collection_name)

  client.create_collection(
      collection_name = collection_name,
      dimension = embedding_dim,
      metric_type = 'IP'
  )

setup_milvus_collection(milvus_client, COLLECTION_NAME ,EMBEDDING_DIM)

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
  File "/usr/local/lib/python3.13/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1357, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


In [16]:
print(milvus_client.list_collections())
print(milvus_client.has_collection(COLLECTION_NAME))

print('Vector Store Initialised')

['hf_documentation']
True
Vector Store Initialised


In [17]:
# defining a function to insert embeddings into the vector store

def insert_data_to_milvus(client: MilvusClient, collection_name: str, chunks: List[Dict],
                          embeddings: List[List[float]], batch_size: int = 100):

  total_inserted = 0

  data = [
      {
      'id': chunk['chunk_id'],
      'vector': embedding,
      'chunk': chunk['chunk'],
      'source': chunk['source']
     } for chunk, embedding in zip(chunks, embeddings)]

  for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i : i+batch_size]
    result = client.insert(collection_name = collection_name, data = batch)
    total_inserted += result['insert_count']

  return total_inserted

In [18]:
inserted_count = insert_data_to_milvus(milvus_client, COLLECTION_NAME, chunks, embeddings)

print(f"\nInserted {inserted_count} records into Milvus")

if inserted_count == len(chunks):
    print("All chunks inserted successfully!")
else:
    print("Not all chunks were inserted. Check your implementation.")

100%|██████████| 275/275 [00:16<00:00, 16.78it/s]


Inserted 27437 records into Milvus
All chunks inserted successfully!


# Retrieval

In [19]:
# defining a retrieval function to retrieve based on similarity search

def retrieve_documents(query: str, client: MilvusClient, collection_name: str,
                       embedding_model: SentenceTransformer, top_k : int = 5) -> List[Dict]:

  retrieved_docs = []

  query_embedding = embedding_model.encode([query], normalize_embeddings = True).tolist()[0]

  search_results = client.search(
      collection_name = collection_name,
      data = [query_embedding],
      limit = top_k,
      search_params={"metric_type": "IP", "params": {}},
      output_fields=["chunk", "source"]
  )

  for result in search_results[0]:
        retrieved_docs.append({
            "text": result["entity"]["chunk"],
            "source": result["entity"]["source"],
            "score": result["distance"]
        })

  return retrieved_docs

In [20]:
# Test retrieval

test_query = "How do I fine-tune a transformer model?"

retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(retrieved)} documents:")
for i, doc in enumerate(retrieved):
    print(f"\n--- Document {i+1} (Score: {doc.get('score', 'N/A')}) ---")
    print(f"Source: {doc.get('source', 'N/A')}")
    print(f"Text: {doc.get('text', 'N/A')[:300]}...")

if len(retrieved) == 3 and all('text' in d for d in retrieved):
    print("\n✅ Retrieval test passed!")
else:
    print("\n❌ Check your retrieval implementation")

Query: How do I fine-tune a transformer model?

Retrieved 3 documents:

--- Document 1 (Score: 0.7614539861679077) ---
Source: huggingface/blog/blob/main/moe.md
Text: on does not degrade quality.">
  <figcaption>Using selective precision does not degrade quality and enables faster models</figcaption>
</figure>

This [notebook](https://colab.research.google.com/drive/1aGGVHZmtKmcNBbAwa9hbu58DDpIuB5O4?usp=sharing) showcases fine-tuning Switch Transformers for summa...

--- Document 2 (Score: 0.7530739307403564) ---
Source: huggingface/blog/blob/main/cv_state.md
Text: our own models

While being able to use a model for off-the-shelf inference is a great way to get started, fine-tuning is where the community gets the most benefits. This is especially true when your datasets are custom, and you’re not getting good performance out of the pre-trained models.

Transfo...

--- Document 3 (Score: 0.7484298348426819) ---
Source: huggingface/blog/blob/main/ray-rag.md
Text: ects/rag/finetune_rag_ra

# Generation

In [21]:
# initialising openai

openai_client = OpenAI(timeout=60, max_retries=3)

LLM_MODEL = 'gpt-4o-mini'

print(f'Using OpenAI Model: {LLM_MODEL}')

Using OpenAI Model: gpt-4o-mini


In [22]:
PROMPT_TEMPLATE = """Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.
If the context doesn't contain enough information to answer the question, say "I don't have enough information to answer this question."

<context>
{context}
</context>

<question>
{question}
</question>

Answer:"""

In [23]:
# defining answer generation

def generate_answer(query: str, retrieved_docs: List[Dict], max_new_tokens: int = 256,
                    temperature: float = 0.7) -> Dict:

    context = "\n\n".join([doc["text"] for doc in retrieved_docs])
    prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9
    )
    answer = response.choices[0].message.content.strip()

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "retrieved_docs": retrieved_docs
    }

In [24]:
# Test generation
test_query = "How do I fine-tune a transformer model?"

# Retrieve relevant documents
retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

# Generate answer
result = generate_answer(
    query=test_query,
    retrieved_docs=retrieved
)

print(f"Question: {result['query']}")
print(f"\nAnswer: {result['answer']}")

if result['answer'] and len(result['answer']) > 10:
    print("\n✅ Generation test passed!")
else:
    print("\n❌ Check your generation implementation")

Question: How do I fine-tune a transformer model?

Answer: To fine-tune a transformer model, you can use the Trainer API provided by the Transformers library. Here are the steps to follow:

1. **Prepare Your Dataset**: Ensure your dataset is ready for the specific task you want to fine-tune your model on.

2. **Choose a Pre-trained Model**: Select a pre-trained transformer model that fits your needs.

3. **Set Up the Trainer**: Utilize the Trainer API which supports various tasks such as image classification, object detection, etc. Ensure that your model computes loss for the task you are fine-tuning.

4. **Fine-tuning Process**: Initiate the fine-tuning process by passing your dataset and model to the Trainer. You can configure hyperparameters to optimize performance.

5. **Hyperparameter Tuning**: Consider using libraries like Ray Tune for scalable and easy hyperparameter tuning, which can significantly impact accuracy.

6. **Monitor Performance**: Keep track of the performance metri

In [25]:
# Complete RAG pipeline function

def rag_query(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    top_k: int = 5,
    max_new_tokens: int = 256,
    temperature: float = 0.7
) -> Dict:
    """
    Complete RAG pipeline: retrieve then generate.
    """
    # Retrieve
    retrieved_docs = retrieve_documents(
        query=query,
        client=client,
        collection_name=collection_name,
        embedding_model=embedding_model,
        top_k=top_k
    )

    # Generate
    result = generate_answer(
        query=query,
        retrieved_docs=retrieved_docs,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )

    return result

In [26]:
# Test complete pipeline with multiple queries

test_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    result = rag_query(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=3
    )
    print(f"Q: {result['query']}")
    print(f"A: {result['answer']}")


Q: What is the Trainer class in transformers?
A: The Trainer class in the Transformers library is a complete training and evaluation loop for PyTorch models. It simplifies the training process by allowing you to pass the necessary components such as the model, tokenizer, dataset, evaluation function, and training hyperparameters, while it takes care of the rest. This makes it easier to start training quickly without the need to manually write a training loop. The Trainer class is also highly customizable, offering a variety of training options to tailor the process to specific needs.

Q: How do I load a dataset from HuggingFace?
A: To load a dataset from Hugging Face, you can use the `load_dataset` function from the 🤗 Datasets library. First, ensure you are logged in with your Hugging Face account using the command:

```bash
huggingface-cli login
```

Then, you can load a dataset by using the following Python code:

```python
from datasets import load_dataset

dataset = load_dataset("

# Evaluation

In [27]:
# defining evaluation metrics

# Opik's default judge model is gpt-5-nano, a reasoning model. It needed around 30 seconds
# per score and kept running into Opik's 60 second request timeout, which is what made the
# evaluation slow and then fail. A fast chat model is set explicitly as the judge instead.
JUDGE_MODEL = 'gpt-4o-mini'

hallucination_metric = Hallucination(model=JUDGE_MODEL, temperature=0)
answer_relevance_metric = AnswerRelevance(model=JUDGE_MODEL, temperature=0)

def evaluate_rag(result: Dict) -> Dict:

  context = [doc['text'] for doc in result['retrieved_docs']]

  hallucination_result = hallucination_metric.score(
      input = result['query'],
      output = result['answer'],
      context = context
  )

  answer_relevance_result = answer_relevance_metric.score(
      input = result['query'],
      output = result['answer'],
      context = context
  )

  return {
        **result,
        "hallucination_score": hallucination_result.value,
        "hallucination_reason": hallucination_result.reason,
        "answer_relevance_score": answer_relevance_result.value,
        "answer_relevance_reason": answer_relevance_result.reason
    }

In [28]:
# writing a function to evaluate the rag pipeline

def evaluate_rag_pipeline(queries: List[str], top_k: int = 3) -> pd.DataFrame:
    eval_results = []

    for query in tqdm(queries):
        result = rag_query(
            query=query,
            client=milvus_client,
            collection_name=COLLECTION_NAME,
            embedding_model=embedding_model,
            top_k=top_k
        )
        scored = evaluate_rag(result)
        eval_results.append({
            "query": scored["query"],
            "answer": scored["answer"],
            "hallucination_score": scored["hallucination_score"],
            "answer_relevance_score": scored["answer_relevance_score"],
        })

    return pd.DataFrame(eval_results)


test_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?",
    "How do I fine-tune a transformer model?"
]

eval_df = evaluate_rag_pipeline(test_queries)
print(eval_df)

print(f"\nAverage hallucination rate: {eval_df['hallucination_score'].mean():.2%}")
print(f"Average answer relevance: {eval_df['answer_relevance_score'].mean():.2f}")

  0%|          | 0/4 [00:00<?, ?it/s]OPIK: No project name configured. Traces are being logged to "Default Project".
Set OPIK_PROJECT_NAME environment variable or pass project_name to the Opik client
to log to a specific project.
See https://www.comet.com/docs/opik/tracing/advanced/sdk_configuration
OPIK: Started logging traces to the "Default Project" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=01a11218-3380-74bb-a6ef-b6fb66cf5797&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.
100%|██████████| 4/4 [00:29<00:00,  7.41s/it]

                                        query  \
0  What is the Trainer class in transformers?   
1   How do I load a dataset from HuggingFace?   
2                    What is Gradio used for?   
3     How do I fine-tune a transformer model?   

                                              answer  hallucination_score  \
0  The Trainer class in the Transformers library ...                  0.0   
1  To load a dataset from Hugging Face, you can u...                  0.0   
2  Gradio is used for creating interactive applic...                  0.0   
3  To fine-tune a transformer model, you can foll...                  0.0   

   answer_relevance_score  
0                    0.95  
1                    1.00  
2                    0.95  
3                    0.95  

Average hallucination rate: 0.00%
Average answer relevance: 0.96


# Evaluation on a Labelled Test Set

The 4 queries above only check the final answer. Here we build a test set of 30 queries where we know which chunk should be retrieved, so we can measure retrieval (Recall@K, MRR) separately from generation (hallucination, answer relevance).

In [29]:
EVAL_FILE = 'eval_queries_v3.json'
NUM_EVAL_QUERIES = 30
EVAL_SEED = 42

QUESTION_PROMPT = """You are writing a test question for a documentation search system.

Read the passage below and write ONE question that a developer could realistically ask,
and that can be answered using only this passage.

Rules:
- The question must name the specific library, model, class, function or feature it is about,
  so it is clear what is being asked without seeing the passage.
- Never refer to "the passage", "the text", "the code", "the example", "the model" or "this approach".
- Ask about a specific fact, step, parameter or concept from the passage.
- Do not copy a full sentence from the passage.
- Write the question in English.
- Return {{"question": "SKIP", "answer": ""}} if the passage is not in English, if it is mostly code,
  a table, links or navigation, or if you cannot write a question that follows these rules.

Return JSON only, in this form: {{"question": "...", "answer": "..."}}

Passage:
{chunk}"""


def is_good_chunk(text: str) -> bool:

  if len(text) < 600:
    return False

  if text.count('|') > 10:
    return False

  # the embedding model is English-only, so skip translated docs
  if sum(ch.isascii() for ch in text) / len(text) < 0.97:
    return False

  letters = sum(ch.isalpha() or ch.isspace() for ch in text)
  return letters / len(text) >= 0.75


def generate_question(chunk_text: str) -> Dict:

  response = openai_client.chat.completions.create(
      model=LLM_MODEL,
      messages=[{"role": "user", "content": QUESTION_PROMPT.format(chunk=chunk_text)}],
      max_tokens=200,
      temperature=0,
      response_format={"type": "json_object"}
  )

  try:
    generated = json.loads(response.choices[0].message.content)
  except (json.JSONDecodeError, TypeError):
    generated = None

  if not isinstance(generated, dict):
    return {"question": "SKIP", "answer": ""}

  return generated


def build_eval_set(chunks: List[Dict], num_queries: int = 30, seed: int = 42) -> List[Dict]:

  rng = random.Random(seed)

  candidates = [chunk for chunk in chunks if is_good_chunk(chunk['chunk'])]
  rng.shuffle(candidates)

  eval_set = []
  used_sources = set()

  # the progress bar counts accepted questions, not candidate chunks
  with tqdm(total=num_queries) as progress:
    for chunk in candidates:
      if len(eval_set) >= num_queries:
        break

      # one query per source document so the test set covers different docs
      if chunk['source'] in used_sources:
        continue

      generated = generate_question(chunk['chunk'])
      question = str(generated.get('question', '')).strip()

      if not question or question.upper() == 'SKIP':
        continue

      eval_set.append({
          'query': question,
          'reference_answer': str(generated.get('answer', '')).strip(),
          'gold_chunk_id': chunk['chunk_id'],
          'gold_source': chunk['source'],
          'gold_chunk_text': chunk['chunk']
      })
      used_sources.add(chunk['source'])
      progress.update(1)

  return eval_set


def is_stale(eval_set: List[Dict], chunks: List[Dict], num_queries: int) -> bool:

  # the saved gold chunk ids are only valid if they still point at the same chunk text
  if len(eval_set) != num_queries:
    return True

  for item in eval_set:
    chunk_id = item['gold_chunk_id']
    if chunk_id >= len(chunks) or chunks[chunk_id]['chunk'] != item.get('gold_chunk_text'):
      return True

  return False


eval_set = None

if os.path.exists(EVAL_FILE):
  with open(EVAL_FILE) as f:
    eval_set = json.load(f)

  if is_stale(eval_set, chunks, NUM_EVAL_QUERIES):
    print(f'{EVAL_FILE} does not match the current chunks or settings, rebuilding it.')
    eval_set = None
  else:
    print(f'Loaded {len(eval_set)} queries from {EVAL_FILE}')

if eval_set is None:
  eval_set = build_eval_set(chunks, NUM_EVAL_QUERIES, EVAL_SEED)
  with open(EVAL_FILE, 'w') as f:
    json.dump(eval_set, f, indent=2)
  print(f'Generated {len(eval_set)} queries and saved them to {EVAL_FILE}')

print()
print('-----Sample Queries-----')
for item in eval_set[:3]:
  print(f"Q: {item['query']}")
  print(f"Gold source: {item['gold_source']} (chunk {item['gold_chunk_id']})")
  print()

100%|██████████| 30/30 [00:42<00:00,  1.40s/it]

Generated 30 queries and saved them to eval_queries_v3.json

-----Sample Queries-----
Q: What function is used to create a question-answering pipeline with the specified model and tokenizer?
Gold source: huggingface/optimum/blob/main/README.md (chunk 23458)

Q: What does the epsilon parameter control in the epsilon_greedy_policy function?
Gold source: huggingface/deep-rl-class/blob/main/units/en/unit2/hands-on.mdx (chunk 15212)

Q: What is the range of values for the Matthews Correlation coefficient?
Gold source: huggingface/evaluate/blob/main/metrics/super_glue/README.md (chunk 3946)



In [30]:
K_VALUES = [1, 3, 5, 10]

def retrieve_with_ids(query: str, client: MilvusClient, collection_name: str,
                      embedding_model: SentenceTransformer, top_k: int = 10) -> List[Dict]:

  query_embedding = embedding_model.encode([query], normalize_embeddings = True).tolist()[0]

  search_results = client.search(
      collection_name = collection_name,
      data = [query_embedding],
      limit = top_k,
      search_params={"metric_type": "IP", "params": {}},
      output_fields=["source"]
  )

  return [{
      'chunk_id': result['id'],
      'source': result['entity']['source'],
      'score': result['distance']
  } for result in search_results[0]]


def first_hit_rank(retrieved: List[Dict], key: str, gold) -> int:
  for rank, doc in enumerate(retrieved, start=1):
    if doc[key] == gold:
      return rank

  return 0


def evaluate_retrieval(eval_set: List[Dict], k_values: List[int] = K_VALUES) -> Tuple[pd.DataFrame, pd.DataFrame]:
  max_k = max(k_values)
  rows = []

  for item in tqdm(eval_set):
    retrieved = retrieve_with_ids(item['query'], milvus_client, COLLECTION_NAME, embedding_model, top_k=max_k)

    rows.append({
        'query': item['query'],
        'gold_source': item['gold_source'],
        'chunk_rank': first_hit_rank(retrieved, 'chunk_id', item['gold_chunk_id']),
        'source_rank': first_hit_rank(retrieved, 'source', item['gold_source'])
    })

  per_query = pd.DataFrame(rows)

  summary = {}
  for level in ['chunk', 'source']:
    ranks = per_query[f'{level}_rank']
    scores = {f'Recall@{k}': ((ranks > 0) & (ranks <= k)).mean() for k in k_values}
    scores[f'MRR@{max_k}'] = ranks.apply(lambda rank: 1 / rank if rank > 0 else 0).mean()
    summary[level] = scores

  return pd.DataFrame(summary).T, per_query


retrieval_summary, retrieval_per_query = evaluate_retrieval(eval_set)

print()
print(f'Retrieval results on {len(eval_set)} queries')
print('chunk  = the exact chunk the question was written from is retrieved')
print('source = any chunk from the correct document is retrieved (easier)')
print()
print(retrieval_summary.round(3))

GENERATION_TOP_K = 3
missed = retrieval_per_query[(retrieval_per_query['chunk_rank'] == 0) | (retrieval_per_query['chunk_rank'] > GENERATION_TOP_K)]

print()
print(f'Queries where the gold chunk is NOT in the top {GENERATION_TOP_K} (what the LLM actually sees): {len(missed)}')
for _, row in missed.iterrows():
  rank = row['chunk_rank'] if row['chunk_rank'] > 0 else f'not in top {max(K_VALUES)}'
  print(f"- [{rank}] {row['query']}")

100%|██████████| 30/30 [00:47<00:00,  1.58s/it]


Retrieval results on 30 queries
chunk  = the exact chunk the question was written from is retrieved
source = any chunk from the correct document is retrieved (easier)

        Recall@1  Recall@3  Recall@5  Recall@10  MRR@10
chunk      0.367     0.467     0.667      0.667   0.447
source     0.500     0.633     0.733      0.733   0.577

Queries where the gold chunk is NOT in the top 3 (what the LLM actually sees): 16
- [not in top 10] What function is used to create a question-answering pipeline with the specified model and tokenizer?
- [not in top 10] What is the range of values for the Matthews Correlation coefficient?
- [5] What is the purpose of the Open LLM Leaderboard?
- [not in top 10] What is the total weight in GB of the IF model when using 32-bit floating point precision?
- [5] What command is used to run browser tests in UI mode?
- [not in top 10] What should be checked in `model.hf_device_map` to avoid CPU-related errors with Linear weights?
- [not in top 10] What is the pur

In [31]:
def evaluate_answers(eval_set: List[Dict], per_query: pd.DataFrame, top_k: int = 3, max_workers: int = 4) -> pd.DataFrame:

  # Step 1: retrieval, one query at a time. It is local and quick, and it keeps the
  # embedding model and Milvus out of the threads below.
  retrieved_per_query = [
      retrieve_documents(
          query=item['query'],
          client=milvus_client,
          collection_name=COLLECTION_NAME,
          embedding_model=embedding_model,
          top_k=top_k
      )
      for item in tqdm(eval_set, desc='Retrieving')
  ]

  # Step 2: generation and scoring. These are OpenAI calls that mostly wait on the network,
  # so a few of them run at the same time.
  def process(job):
    item, chunk_rank, retrieved_docs = job

    row = {
        'query': item['query'],
        'answer': '',
        'reference_answer': item['reference_answer'],
        'gold_in_context': bool(0 < chunk_rank <= top_k),
        'hallucination_score': np.nan,
        'answer_relevance_score': np.nan
    }

    try:
      result = generate_answer(query=item['query'], retrieved_docs=retrieved_docs, temperature=0)
      row['answer'] = result['answer']

      scored = evaluate_rag(result)
      row['hallucination_score'] = scored['hallucination_score']
      row['answer_relevance_score'] = scored['answer_relevance_score']
    except Exception as error:
      print(f"Failed for: {item['query']} ({error})")

    return row

  jobs = list(zip(eval_set, per_query['chunk_rank'], retrieved_per_query))

  with ThreadPoolExecutor(max_workers=max_workers) as pool:
    rows = list(tqdm(pool.map(process, jobs), total=len(jobs), desc='Answering and scoring'))

  return pd.DataFrame(rows)


answer_df = evaluate_answers(eval_set, retrieval_per_query, top_k=GENERATION_TOP_K)
answer_df.to_csv('eval_results.csv', index=False)

scored_df = answer_df.dropna(subset=['hallucination_score', 'answer_relevance_score'])
not_scored = len(answer_df) - len(scored_df)

print()
print(f'Answer quality on {len(scored_df)} of {len(answer_df)} queries (top_k = {GENERATION_TOP_K})')
if not_scored:
  print(f'WARNING: {not_scored} queries could not be scored and are left out of the numbers below.')
print(f"Average hallucination score: {scored_df['hallucination_score'].mean():.2f}")
print(f"Average answer relevance: {scored_df['answer_relevance_score'].mean():.2f}")

print()
print('Split by whether the gold chunk was in the context:')
print(scored_df.groupby('gold_in_context').agg(
    queries=('query', 'count'),
    hallucination=('hallucination_score', 'mean'),
    answer_relevance=('answer_relevance_score', 'mean')
).round(2))

LOW_RELEVANCE = 0.7
weak = scored_df[scored_df['answer_relevance_score'] < LOW_RELEVANCE]

print()
print(f'Answers with relevance below {LOW_RELEVANCE}: {len(weak)}')
for _, row in weak.iterrows():
  cause = 'generation issue (gold chunk was in context)' if row['gold_in_context'] else 'retrieval miss'
  print(f"- {row['answer_relevance_score']:.2f} | {cause} | {row['query']}")

Answering and scoring: 100%|██████████| 30/30 [00:31<00:00,  1.04s/it]


Answer quality on 30 of 30 queries (top_k = 3)
Average hallucination score: 0.03
Average answer relevance: 0.72

Split by whether the gold chunk was in the context:
                 queries  hallucination  answer_relevance
gold_in_context                                          
False                 16           0.06              0.48
True                  14           0.00              0.99

Answers with relevance below 0.7: 9
- 0.10 | retrieval miss | What function is used to create a question-answering pipeline with the specified model and tokenizer?
- 0.10 | retrieval miss | What is the total weight in GB of the IF model when using 32-bit floating point precision?
- 0.10 | retrieval miss | What should be checked in `model.hf_device_map` to avoid CPU-related errors with Linear weights?
- 0.10 | retrieval miss | What is the purpose of the StableDiffusionAttendAndExcitePipeline class?
- 0.10 | retrieval miss | What is the shortname for the benchmark used to measure Massive Multitas